# 🔐 Python Dotenv Tutorial
## Managing Environment Variables Securely

**Date:** April 26, 2026  
**Topic:** Environment Variables & Security Best Practices  
**Why This Matters:** Prevent credential leaks, manage configurations professionally

---

## 📚 What I Actually Learn

1. Why environment variables are important
2. How to use python-dotenv library
3. Best practices for managing secrets
4. Real-world examples (database, API keys, etc.)

---

## ⚠️ The Problem: Hardcoded Credentials

### ❌ BAD PRACTICE (Never do this!):

```python
# NEVER hardcode credentials!
database_password = "mySecretPassword123"  # ❌ DANGER!
api_key = "sk-1234567890abcccdef"           # ❌ Will leak if pushed to GitHub!
secret_token = "super_secret_token"       # ❌ Security risk!
```

### 🚨 Why This Is Dangerous:

1. **GitHub is public** - Anyone can see your secrets
2. **Git history** - Even if you delete it, it's in commit history
3. **Bots scan GitHub** - Automated tools steal exposed credentials
4. **Financial loss** - Stolen API keys = unauthorized charges
5. **Data breach** - Database credentials = full data access

### 📰 Real Examples:
- AWS keys leaked → $50,000 bill in 24 hours
- Database credentials exposed → customer data stolen
- API tokens leaked → service abuse and suspension

---

## ✅ The Solution: Environment Variables with Dotenv

### Step 1: Install python-dotenv

```bash
pip install python-dotenv
```

In [ ]:
# Install in Jupyter
!pip install python-dotenv

### Step 2: Create .env File

Create a file named `.env` in your project root:

```env
# Database Configuration
DB_HOST=localhost
DB_PORT=5432
DB_NAME=myDatabase
DB_USER=admin
DB_PASSWORD=SuperSecretPassword123!

# API Keys
OPENAI_API_KEY=sk-1234567890abcdef
STRIPE_SECRET_KEY=sk_test_xxxxxxxxxx

# Application Settings
DEBUG_MODE=True
SECRET_KEY=django-secret-key-here
```

**IMPORTANT:** Add `.env` to `.gitignore` immediately!

### Step 3: Load Environment Variables

In [ ]:
# Import required libraries
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Access environment variables
db_host = os.getenv('DB_HOST')
db_password = os.getenv('DB_PASSWORD')
api_key = os.getenv('OPENAI_API_KEY')

print(f"Database Host: {db_host}")
print(f"Password: {'*' * len(db_password) if db_password else 'Not set'}")
print(f"API Key: {api_key[:8]}...{api_key[-4:] if api_key else 'Not set'}")

## 🎯 Example 1: Database Connection

In [ ]:
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

class DatabaseConfig:
    """Secure database configuration using environment variables"""
    
    def __init__(self):
        self.host = os.getenv('DB_HOST', 'localhost')  # Default: localhost
        self.port = int(os.getenv('DB_PORT', 5432))    # Default: 5432
        self.database = os.getenv('DB_NAME')
        self.user = os.getenv('DB_USER')
        self.password = os.getenv('DB_PASSWORD')
    
    def get_connection_string(self):
        """Generate database connection string"""
        return f"postgresql://{self.user}:{self.password}@{self.host}:{self.port}/{self.database}"
    
    def validate(self):
        """Validate that all required variables are set"""
        required = ['database', 'user', 'password']
        missing = [field for field in required if not getattr(self, field)]
        
        if missing:
            raise ValueError(f"Missing environment variables: {', '.join(missing)}")
        return True

# Usage
db_config = DatabaseConfig()

try:
    db_config.validate()
    print("✅ Database configuration valid!")
    print(f"Host: {db_config.host}")
    print(f"Database: {db_config.database}")
    # Don't print actual connection string in production!
except ValueError as e:
    print(f"❌ Configuration error: {e}")

## 🎯 Example 2: API Key Management

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

class APIClient:
    """Generic API client with secure key management"""
    
    def __init__(self, service_name):
        self.service = service_name
        self.api_key = os.getenv(f'{service_name.upper()}_API_KEY')
        
        if not self.api_key:
            raise ValueError(f"API key for {service_name} not found in environment")
    
    def make_request(self, endpoint):
        """Simulate API request"""
        # In real code, this would use requests library
        print(f"Making request to {self.service} API...")
        print(f"Endpoint: {endpoint}")
        print(f"Using API Key: {self.api_key[:8]}...{self.api_key[-4:]}")
        return {"status": "success", "message": "Request completed"}

# Usage examples
try:
    # OpenAI client
    openai_client = APIClient('openai')
    response = openai_client.make_request('/v1/completions')
    print(f"Response: {response}\n")
    
    # Stripe client
    stripe_client = APIClient('stripe_secret')
    response = stripe_client.make_request('/v1/charges')
    print(f"Response: {response}")
    
except ValueError as e:
    print(f"❌ Error: {e}")

## 🎯 Example 3: Application Configuration

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

class AppConfig:
    """Application configuration management"""
    
    # Environment
    ENV = os.getenv('ENVIRONMENT', 'development')
    DEBUG = os.getenv('DEBUG_MODE', 'False').lower() == 'true'
    
    # Security
    SECRET_KEY = os.getenv('SECRET_KEY')
    
    # Database
    DATABASE_URL = os.getenv('DATABASE_URL')
    
    # External Services
    REDIS_URL = os.getenv('REDIS_URL', 'redis://localhost:6379')
    
    # Feature Flags
    ENABLE_ANALYTICS = os.getenv('ENABLE_ANALYTICS', 'False').lower() == 'true'
    ENABLE_CACHING = os.getenv('ENABLE_CACHING', 'True').lower() == 'true'
    
    @classmethod
    def display_config(cls):
        """Display current configuration (safe for logging)"""
        print("=" * 50)
        print("APPLICATION CONFIGURATION")
        print("=" * 50)
        print(f"Environment: {cls.ENV}")
        print(f"Debug Mode: {cls.DEBUG}")
        print(f"Secret Key: {'*' * 20} (hidden)")
        print(f"Database: {'Configured' if cls.DATABASE_URL else 'Not set'}")
        print(f"Redis: {cls.REDIS_URL}")
        print(f"Analytics: {'Enabled' if cls.ENABLE_ANALYTICS else 'Disabled'}")
        print(f"Caching: {'Enabled' if cls.ENABLE_CACHING else 'Disabled'}")
        print("=" * 50)

# Display configuration
AppConfig.display_config()

## 🎯 Example 4: Multiple Environment Files

In [ ]:
from dotenv import load_dotenv
import os

# Load different .env files based on environment
environment = os.getenv('ENVIRONMENT', 'development')

if environment == 'production':
    load_dotenv('.env.production')
    print("✅ Loaded PRODUCTION environment")
elif environment == 'staging':
    load_dotenv('.env.staging')
    print("✅ Loaded STAGING environment")
else:
    load_dotenv('.env.development')
    print("✅ Loaded DEVELOPMENT environment")

print(f"Current environment: {environment}")
print(f"Debug mode: {os.getenv('DEBUG_MODE')}")

## 🛡️ Security Best Practices

### ✅ DO:

1. **Always add `.env` to `.gitignore`**
   ```gitignore
   .env
   .env.local
   .env.*.local
   ```

2. **Create `.env.example` template**
   ```env
   # .env.example - Safe to commit
   DB_HOST=localhost
   DB_PORT=5432
   DB_NAME=your_database_name
   DB_USER=your_username
   DB_PASSWORD=your_password_here
   ```

3. **Use default values for non-sensitive configs**
   ```python
   DEBUG = os.getenv('DEBUG', 'False')
   PORT = int(os.getenv('PORT', 8000))
   ```

4. **Validate required variables on startup**
   ```python
   required_vars = ['DB_PASSWORD', 'SECRET_KEY', 'API_KEY']
   for var in required_vars:
       if not os.getenv(var):
           raise ValueError(f"Missing required environment variable: {var}")
   ```

5. **Use different .env files for different environments**
   - `.env.development`
   - `.env.staging`
   - `.env.production`

### ❌ DON'T:

1. **Never commit .env files**
2. **Don't hardcode secrets as fallbacks**
   ```python
   # ❌ BAD
   password = os.getenv('PASSWORD', 'default_password')
   ```

3. **Don't print secrets to logs**
   ```python
   # ❌ BAD
   print(f"Password: {os.getenv('PASSWORD')}")
   ```

4. **Don't use .env in production (use proper secret management)**
   - AWS Secrets Manager
   - Azure Key Vault
   - HashiCorp Vault
   - Environment variables from hosting platform

---

## 🎯 Real-World Data Science Example

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

class DataPipeline:
    """Data pipeline with secure configuration"""
    
    def __init__(self):
        # Load credentials from environment
        self.db_connection = os.getenv('DATABASE_URL')
        self.api_key = os.getenv('DATA_API_KEY')
        self.output_path = os.getenv('OUTPUT_PATH', './data/output')
        
        # Validate configuration
        if not all([self.db_connection, self.api_key]):
            raise ValueError("Missing required credentials in .env file")
    
    def extract_data(self):
        """Extract data from database"""
        print("🔄 Extracting data from database...")
        # In real code: pd.read_sql(query, self.db_connection)
        # Simulated data
        data = pd.DataFrame({
            'id': [1, 2, 3],
            'value': [100, 200, 300]
        })
        print("✅ Data extracted successfully")
        return data
    
    def transform_data(self, data):
        """Transform data"""
        print("🔄 Transforming data...")
        data['value_doubled'] = data['value'] * 2
        print("✅ Data transformed successfully")
        return data
    
    def load_data(self, data):
        """Load data to destination"""
        print(f"🔄 Loading data to {self.output_path}...")
        # In real code: data.to_csv(f"{self.output_path}/result.csv")
        print("✅ Data loaded successfully")
        return True
    
    def run(self):
        """Run complete ETL pipeline"""
        print("\n" + "="*50)
        print("STARTING DATA PIPELINE")
        print("="*50 + "\n")
        
        data = self.extract_data()
        data = self.transform_data(data)
        self.load_data(data)
        
        print("\n" + "="*50)
        print("✅ PIPELINE COMPLETED SUCCESSFULLY")
        print("="*50)
        
        return data

# Run pipeline
try:
    pipeline = DataPipeline()
    result = pipeline.run()
    print("\nResult preview:")
    print(result.head())
except ValueError as e:
    print(f"❌ Configuration Error: {e}")
    print("Please check your .env file and ensure all required variables are set.")

## 📝 Summary

### Key Takeaways:

1. **Never hardcode credentials** - Use environment variables
2. **Always use .gitignore** - Prevent .env from being committed
3. **Create .env.example** - Template for other developers
4. **Validate on startup** - Fail fast if configuration is missing
5. **Use proper secret management in production** - .env is for development

### Real-World Applications:

- 🗄️ **Database connections** (PostgreSQL, MySQL, MongoDB)
- 🔑 **API keys** (OpenAI, Stripe, AWS, Google Cloud)
- 🔐 **Authentication tokens** (JWT secrets, OAuth credentials)
- ⚙️ **Configuration** (debug mode, feature flags)
- 📊 **Data pipelines** (source/destination credentials)

### Next Steps:

1. Install python-dotenv in your projects
2. Move all hardcoded credentials to .env
3. Update .gitignore to exclude .env
4. Create .env.example as documentation
5. Use environment-specific .env files

---

**Remember:** Security is not optional. Protecting credentials is your responsibility as a developer. 🛡️

---